<a href="https://colab.research.google.com/github/MoeTahech/real-fake-face-detection-thesis/blob/main/notebooks/02b_ablation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02b — Ablation: separating the effect of unfreezing from the effect of learning rate
Baseline #1 (frozen, lr 1e-3) and Phase 2 (layer4 unfrozen, lr 1e-4) differ in **two** things at once.
This notebook changes **one factor at a time**, using the exact same data, augmentation, optimizer,
batch size and number of epochs as Phase 2.

| Run | Trainable | LR | Purpose |
|---|---|---|---|
| Baseline #1 | final layer | 1e-3 | already done |
| **A** | final layer | **1e-4** | isolates the effect of the learning rate (the important one) |
| **B** (optional) | **layer4 + final layer** | **1e-3** | isolates the effect of unfreezing |
| Phase 2 | layer4 + final layer | 1e-4 | already done |

Runtime → Change runtime type → GPU, then run cells top to bottom.

### Step 1 — GPU

In [ ]:
!nvidia-smi

Fri Oct 16 12:00:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Step 2 — Mount Drive (checkpoints are saved under new names, nothing is overwritten)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/thesis_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("Checkpoints dir:", CHECKPOINT_DIR)

Mounted at /content/drive
Checkpoints dir: /content/drive/MyDrive/thesis_checkpoints


### Step 3 — Kaggle token (do not share the notebook with a real token in it)

In [ ]:
import os
os.environ["KAGGLE_API_TOKEN"] = "YOUR_KAGGLE_TOKEN_HERE"

### Step 4 — Download and link the dataset (same as notebook 02)

In [ ]:
!pip install -q kaggle
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces
!unzip -q 140k-real-and-fake-faces.zip -d data_raw

import os
base_abs = os.path.abspath("data_raw/real_vs_fake/real-vs-fake")
os.makedirs("data", exist_ok=True)
!ln -sfn "{base_abs}/train" data/train
!ln -sfn "{base_abs}/valid" data/val

print("train/real:", len(os.listdir("data/train/real")), "| train/fake:", len(os.listdir("data/train/fake")))
print("val/real:", len(os.listdir("data/val/real")), "| val/fake:", len(os.listdir("data/val/fake")))

Dataset URL: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
License(s): other
100% 3.75G/3.75G [03:46<00:00, 17.8MB/s]

train/real: 50000 | train/fake: 50000
val/real: 10000 | val/fake: 10000


### Step 5 — Training script (same as Phase 2, with two switches added)

In [ ]:
%%writefile train_ablation.py
"""
Ablation version of train_finetune.py. Identical in every respect (data, augmentation,
optimizer, batch size, epochs) EXCEPT two switches:
    --unfreeze_layer4 {0,1}   whether layer4 is trainable
    --lr                      learning rate
    --ckpt_name               checkpoint filename (so nothing gets overwritten)
"""
import argparse, os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from tqdm import tqdm


def get_dataloaders(data_dir, batch_size, img_size=224):
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    train_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ])
    val_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        normalize,
    ])
    train_set = datasets.ImageFolder(f"{data_dir}/train", transform=train_transform)
    val_set = datasets.ImageFolder(f"{data_dir}/val", transform=val_transform)
    print("Class mapping:", train_set.class_to_idx)
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=2)
    return train_loader, val_loader, train_set.class_to_idx


def build_model(unfreeze_layer4):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    for param in model.parameters():
        param.requires_grad = False
    if unfreeze_layer4:
        for param in model.layer4.parameters():
            param.requires_grad = True
    model.fc = nn.Linear(model.fc.in_features, 2)
    return model


def run_epoch(model, loader, criterion, optimizer, device, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for images, labels in tqdm(loader, desc="train" if train else "val"):
            images, labels = images.to(device), labels.to(device)
            if train:
                optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_dir", type=str, default="./data")
    parser.add_argument("--epochs", type=int, default=5)
    parser.add_argument("--batch_size", type=int, default=64)
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--unfreeze_layer4", type=int, default=0)
    parser.add_argument("--ckpt_name", type=str, required=True)
    parser.add_argument("--checkpoint_dir", type=str, default=".")
    parser.add_argument("--seed", type=int, default=0)
    args = parser.parse_args()

    torch.manual_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)
    print(f"Config: unfreeze_layer4={args.unfreeze_layer4} lr={args.lr} epochs={args.epochs} seed={args.seed}")

    train_loader, val_loader, class_to_idx = get_dataloaders(args.data_dir, args.batch_size)
    model = build_model(bool(args.unfreeze_layer4)).to(device)
    criterion = nn.CrossEntropyLoss()

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    n_trainable = sum(p.numel() for p in trainable_params)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"Trainable parameters: {n_trainable:,} / {n_total:,}")

    optimizer = torch.optim.Adam(trainable_params, lr=args.lr)

    os.makedirs(args.checkpoint_dir, exist_ok=True)
    save_path = os.path.join(args.checkpoint_dir, args.ckpt_name)

    best_val_acc = 0.0
    for epoch in range(args.epochs):
        print(f"\nEpoch {epoch + 1}/{args.epochs}")
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, device, train=True)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer, device, train=False)
        print(f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({"model_state": model.state_dict(), "class_to_idx": class_to_idx}, save_path)
            print(f"Saved new best model to {save_path} (val_acc={val_acc:.4f})")

    print(f"\nDone. Best validation accuracy: {best_val_acc:.4f}")


if __name__ == "__main__":
    main()

Writing train_ablation.py


### Step 6 — Run A: frozen backbone, lr 1e-4  *(the important one)*
If this stays far below Phase 2 (98.8%), unfreezing `layer4` is what made the difference.
If it gets close, the learning rate explained a lot of the gain.

In [ ]:
!python train_ablation.py --data_dir ./data --epochs 5 --batch_size 64 --lr 1e-4 --unfreeze_layer4 0 --ckpt_name ablation_frozen_lr1e-4.pth --checkpoint_dir "{CHECKPOINT_DIR}"

Using device: cuda
Config: unfreeze_layer4=0 lr=0.0001 epochs=5 seed=0
Class mapping: {'fake': 0, 'real': 1}
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 167MB/s]
Trainable parameters: 1,026 / 11,177,538

Epoch 1/5
train: 100% 1563/1563 [05:26<00:00,  4.79it/s]
val: 100% 313/313 [00:59<00:00,  5.27it/s]
train_loss=0.5341 train_acc=0.7352 val_loss=0.4914 val_acc=0.7616
Saved new best model to /content/drive/MyDrive/thesis_checkpoints/ablation_frozen_lr1e-4.pth (val_acc=0.7616)

Epoch 2/5
train: 100% 1563/1563 [04:53<00:00,  5.33it/s]
val: 100% 313/313 [00:56<00:00,  5.51it/s]
train_loss=0.4755 train_acc=0.7768 val_loss=0.4612 val_acc=0.7847
Saved new best model to /content/drive/MyDrive/thesis_checkpoints/ablation_frozen_lr1e-4.pth (val_acc=0.7847)

Epoch 3/5
train: 100% 1563/1563 [04:48<00:00,  5.42it/s]
val: 100% 313/313 [00:55<00:00,  5.66it/s]
train_loss=0.4573 trai

### Step 7 (optional) — Run B: layer4 unfrozen, lr 1e-3
Completes the 2×2 table. Skip it if you are short on Colab GPU time.

In [ ]:
!python train_ablation.py --data_dir ./data --epochs 5 --batch_size 64 --lr 1e-3 --unfreeze_layer4 1 --ckpt_name ablation_layer4_lr1e-3.pth --checkpoint_dir "{CHECKPOINT_DIR}"

Using device: cuda
Config: unfreeze_layer4=1 lr=0.001 epochs=5 seed=0
Class mapping: {'fake': 0, 'real': 1}
Trainable parameters: 8,394,754 / 11,177,538

Epoch 1/5
train: 100% 1563/1563 [05:01<00:00,  5.19it/s]
val: 100% 313/313 [00:55<00:00,  5.60it/s]
train_loss=0.1517 train_acc=0.9390 val_loss=0.0754 val_acc=0.9711
Saved new best model to /content/drive/MyDrive/thesis_checkpoints/ablation_layer4_lr1e-3.pth (val_acc=0.9711)

Epoch 2/5
train: 100% 1563/1563 [04:49<00:00,  5.40it/s]
val: 100% 313/313 [00:56<00:00,  5.53it/s]
train_loss=0.0662 train_acc=0.9748 val_loss=0.0531 val_acc=0.9797
Saved new best model to /content/drive/MyDrive/thesis_checkpoints/ablation_layer4_lr1e-3.pth (val_acc=0.9797)

Epoch 3/5
train: 100% 1563/1563 [04:55<00:00,  5.28it/s]
val: 100% 313/313 [00:54<00:00,  5.71it/s]
train_loss=0.0447 train_acc=0.9834 val_loss=0.0448 val_acc=0.9840
Saved new best model to /content/drive/MyDrive/thesis_checkpoints/ablation_layer4_lr1e-3.pth (val_acc=0.9840)

Epoch 4/5
train

### Step 8 — Evaluate on the full test set and compare everything

In [ ]:
import torch, torch.nn as nn, os
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, roc_auc_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
test_transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), normalize])
test_set = datasets.ImageFolder("data_raw/real_vs_fake/real-vs-fake/test", transform=test_transform)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False, num_workers=2)

def evaluate(ckpt_name):
    path = f"{CHECKPOINT_DIR}/{ckpt_name}"
    if not os.path.exists(path):
        return None
    ckpt = torch.load(path, map_location=device)
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, 2)
    model.load_state_dict(ckpt["model_state"])
    model.to(device).eval()
    labels_all, probs_all, preds_all = [], [], []
    with torch.no_grad():
        for images, labels in test_loader:
            logits = model(images.to(device))
            probs_all.extend(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
            preds_all.extend(logits.argmax(dim=1).cpu().numpy())
            labels_all.extend(labels.numpy())
    return accuracy_score(labels_all, preds_all), roc_auc_score(labels_all, probs_all)

rows = [
    ("Baseline #1 (notebook 01)", "frozen", "1e-3", (0.8077, 0.8883)),
    ("Ablation A", "frozen", "1e-4", evaluate("ablation_frozen_lr1e-4.pth")),
    ("Ablation B (optional)", "layer4", "1e-3", evaluate("ablation_layer4_lr1e-3.pth")),
    ("Phase 2 (notebook 02)", "layer4", "1e-4", (0.9881, 0.9993)),
]

print(f"{'Run':<28}{'Trainable':<11}{'LR':<7}{'Test Acc':<10}{'Test AUC'}")
print("-" * 64)
for name, trainable, lr, res in rows:
    if res is None:
        print(f"{name:<28}{trainable:<11}{lr:<7}{'(not run)':<10}")
    else:
        print(f"{name:<28}{trainable:<11}{lr:<7}{res[0]:<10.4f}{res[1]:.4f}")

---
### How to read the table
- **A ≈ Baseline** → the learning rate barely mattered; the gain comes from unfreezing `layer4`.
- **A ≈ Phase 2** → the learning rate (not unfreezing) explained most of the gain.
- **In between** → both contribute; report both honestly.
- If B was run: **B vs Phase 2** shows how much a higher learning rate hurts or helps when `layer4` is trainable.

### Caveats to mention in the report
- Each configuration was run **once (seed 0)**, so very small differences are not reliable.
- Frozen layers' BatchNorm running statistics are still updated in `model.train()` (same as in notebooks 01 and 02, so the comparison stays consistent).
- All numbers are on the same-distribution test set; cross-dataset results come from notebook 03.